# Practice 1 - CNNs: Part 1 - Custom CNNs

**Authors:** [Name 1], [Name 2]

**Date:** March 2026

**Description:** In this notebook we develop several custom convolutional neural network (CNN) models to solve the classification problem posed by the STL-10 dataset.

---
## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Dataset Loading & Exploration](#2-dataset-loading--exploration)
3. [Data Preprocessing](#3-data-preprocessing)
4. [Data Augmentation](#4-data-augmentation)
5. [Baseline CNN Model](#5-baseline-cnn-model)
6. [Improved CNN Model](#6-improved-cnn-model)
7. [Advanced Architecture: Residual Network](#7-advanced-architecture-residual-network)
8. [Advanced Architecture: Inception / Xception](#8-advanced-architecture-inception--xception)
9. [Results Comparison](#9-results-comparison)
10. [Conclusions](#10-conclusions)

---
## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers, Model
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Constants
IMG_SIZE = 96        # STL-10 native resolution
NUM_CLASSES = 10
BATCH_SIZE = 32
EPOCHS = 50          # Adjust as needed
VALIDATION_SPLIT = 0.2

# Directory to save models
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

---
## 2. Dataset Loading & Exploration

The **STL-10** dataset contains:
- 5,000 labeled training images (500 per class)
- 8,000 labeled test images (800 per class)
- 100,000 unlabeled images (not used here)
- Image size: 96×96 pixels, 3 color channels (RGB)
- 10 classes: airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck

In [ ]:
# Load STL-10 from TensorFlow Datasets
(ds_train, ds_test), ds_info = tfds.load(
    'stl10',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True
)

print(ds_info)

In [ ]:
# Class names
class_names = ds_info.features['label'].names
print(f"Classes: {class_names}")
print(f"Number of training samples: {ds_info.splits['train'].num_examples}")
print(f"Number of test samples: {ds_info.splits['test'].num_examples}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, (image, label) in enumerate(ds_train.take(10)):
    ax = axes[i // 5, i % 5]
    ax.imshow(image.numpy())
    ax.set_title(class_names[label.numpy()])
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Check class distribution
train_labels = [label.numpy() for _, label in ds_train]
unique, counts = np.unique(train_labels, return_counts=True)

plt.figure(figsize=(10, 4))
plt.bar([class_names[u] for u in unique], counts, color='steelblue')
plt.title('Training Set Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing

Steps:
- Normalize pixel values to [0, 1]
- Resize images if needed (STL-10 is already 96×96)
- Create a validation split from the training set
- Build efficient `tf.data` pipelines with prefetching

In [ ]:
def preprocess(image, label):
    """Normalize images to [0, 1] and resize."""
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

In [ ]:
# Total training samples
num_train = ds_info.splits['train'].num_examples
num_val = int(num_train * VALIDATION_SPLIT)
num_train_actual = num_train - num_val

print(f"Training samples: {num_train_actual}")
print(f"Validation samples: {num_val}")

# Shuffle and split into train/validation
ds_train_shuffled = ds_train.shuffle(num_train, seed=SEED)

ds_val = ds_train_shuffled.take(num_val)
ds_train_split = ds_train_shuffled.skip(num_val)

# Apply preprocessing and batching
train_dataset = (
    ds_train_split
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(num_train_actual)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    ds_val
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    ds_test
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# Verify shapes
for images, labels in train_dataset.take(1):
    print(f"Batch images shape: {images.shape}")
    print(f"Batch labels shape: {labels.shape}")
    print(f"Pixel value range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")

---
## 4. Data Augmentation

To combat overfitting (especially with only ~4,000 training samples), we define data augmentation layers that will be embedded in the model and only active during training.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

In [ ]:
# Visualize augmented images
plt.figure(figsize=(10, 10))
for images, _ in train_dataset.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy())
        plt.axis('off')
plt.suptitle('Augmented Image Examples', fontsize=14)
plt.tight_layout()
plt.show()

---
## Helper Functions

Utility functions for training, evaluation, and plotting that will be reused across models.

In [ ]:
def plot_history(history, title=""):
    """Plot training and validation accuracy and loss curves."""
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs_range, acc, 'b-o', label='Training Accuracy')
    ax1.plot(epochs_range, val_acc, 'r--o', label='Validation Accuracy')
    ax1.set_title(f'{title} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, loss, 'b-o', label='Training Loss')
    ax2.plot(epochs_range, val_loss, 'r--o', label='Validation Loss')
    ax2.set_title(f'{title} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def train_and_evaluate(model, model_name, train_ds, val_ds, test_ds, epochs=EPOCHS):
    """Train a model, plot history, evaluate on test set, and return results."""
    
    filepath = os.path.join(MODEL_DIR, f"{model_name}.keras")
    
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=filepath,
            save_best_only=True,
            monitor='val_loss',
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )
    ]

    history = model.fit(
        train_ds,
        epochs=epochs,
        validation_data=val_ds,
        callbacks=callbacks
    )

    # Plot training curves
    plot_history(history, title=model_name)

    # Evaluate on test set
    best_model = keras.models.load_model(filepath)
    test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
    print(f"\n{model_name} - Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

    return {
        'model_name': model_name,
        'history': history,
        'test_accuracy': test_acc,
        'test_loss': test_loss
    }

In [ ]:
# Store results for comparison
results = []

---
## 5. Baseline CNN Model

A simple VGG-style architecture with stacked Conv2D + MaxPooling2D blocks. This serves as our baseline to benchmark improvements.

**Architecture:**
- 3 convolutional blocks (Conv2D → MaxPooling2D)
- Flatten → Dense → Output
- No data augmentation, no regularization

In [ ]:
def build_baseline_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """Build a simple baseline CNN (no augmentation, no regularization)."""
    inputs = keras.Input(shape=input_shape)
    
    # Feature learning
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Classification
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='baseline_cnn')
    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()

In [ ]:
baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_baseline = train_and_evaluate(
    baseline_model, 'baseline_cnn',
    train_dataset, val_dataset, test_dataset
)
results.append(result_baseline)

**Observations (Baseline):**

_TODO: Comment on the training curves. Is there overfitting? What accuracy did we achieve? Why is this expected given the small training set?_

---
## 6. Improved CNN Model

Improvements over the baseline:
- **Data augmentation** to increase effective training samples
- **Dropout** for regularization
- **Batch Normalization** for faster convergence
- **Deeper architecture** with more filters

In [ ]:
def build_improved_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """Build an improved CNN with augmentation, batch norm, and dropout."""
    inputs = keras.Input(shape=input_shape)
    
    # Data augmentation (only active during training)
    x = data_augmentation(inputs)
    
    # Block 1
    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Block 2
    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Block 3
    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Block 4
    x = layers.Conv2D(256, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(256, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Classification
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='improved_cnn')
    return model

improved_model = build_improved_cnn()
improved_model.summary()

In [ ]:
improved_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_improved = train_and_evaluate(
    improved_model, 'improved_cnn',
    train_dataset, val_dataset, test_dataset
)
results.append(result_improved)

**Observations (Improved CNN):**

_TODO: Compare with the baseline. Has overfitting been reduced? Did accuracy improve? Comment on the effect of data augmentation, batch normalization, and dropout._

---
## 7. Advanced Architecture: Residual Network

We implement a custom **ResNet-style** model with skip connections. Residual connections help with training deeper networks by alleviating the vanishing gradient problem.

**Key idea:** Instead of learning $H(x)$, the network learns the residual $F(x) = H(x) - x$, so the output becomes $F(x) + x$.

In [ ]:
def residual_block(x, filters, downsample=False):
    """A residual block with optional downsampling."""
    strides = 2 if downsample else 1
    shortcut = x
    
    # Main path
    x = layers.Conv2D(filters, 3, strides=strides, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut path: adjust dimensions if needed
    if downsample or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=strides, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add shortcut
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

In [ ]:
def build_custom_resnet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """Build a custom ResNet-style model."""
    inputs = keras.Input(shape=input_shape)
    
    # Data augmentation
    x = data_augmentation(inputs)
    
    # Initial convolution
    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Residual blocks
    x = residual_block(x, 32)
    x = residual_block(x, 64, downsample=True)
    x = residual_block(x, 64)
    x = residual_block(x, 128, downsample=True)
    x = residual_block(x, 128)
    x = residual_block(x, 256, downsample=True)
    x = residual_block(x, 256)
    
    # Global Average Pooling (replaces Flatten + Dense)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='custom_resnet')
    return model

resnet_model = build_custom_resnet()
resnet_model.summary()

In [ ]:
resnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_resnet = train_and_evaluate(
    resnet_model, 'custom_resnet',
    train_dataset, val_dataset, test_dataset
)
results.append(result_resnet)

**Observations (Custom ResNet):**

_TODO: Comment on how residual connections affect training. Did the deeper network train well? Compare with previous models._

---
## 8. Advanced Architecture: Inception / Xception

We implement a custom model inspired by **Inception** / **Xception** architectures:
- **Inception module:** Applies multiple filter sizes (1×1, 3×3, 5×5) in parallel and concatenates the outputs.
- **Depthwise separable convolutions** (Xception-style): More parameter-efficient than standard convolutions.

In [ ]:
def inception_module(x, filters_1x1, filters_3x3, filters_5x5):
    """A simplified Inception module with parallel convolutions."""
    # 1x1 branch
    branch_1x1 = layers.Conv2D(filters_1x1, 1, padding='same', activation='relu')(x)
    
    # 3x3 branch
    branch_3x3 = layers.Conv2D(filters_3x3, 3, padding='same', activation='relu')(x)
    
    # 5x5 branch
    branch_5x5 = layers.Conv2D(filters_5x5, 5, padding='same', activation='relu')(x)
    
    # Max pooling branch
    branch_pool = layers.MaxPooling2D(3, strides=1, padding='same')(x)
    branch_pool = layers.Conv2D(filters_1x1, 1, padding='same', activation='relu')(branch_pool)
    
    # Concatenate
    output = layers.Concatenate()([branch_1x1, branch_3x3, branch_5x5, branch_pool])
    return output

In [ ]:
def build_custom_inception(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """Build a custom Inception-style model."""
    inputs = keras.Input(shape=input_shape)
    
    # Data augmentation
    x = data_augmentation(inputs)
    
    # Initial convolution
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Inception modules
    x = inception_module(x, 32, 48, 16)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = inception_module(x, 64, 96, 32)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = inception_module(x, 128, 192, 64)
    x = layers.BatchNormalization()(x)
    
    # Classification
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='custom_inception')
    return model

inception_model = build_custom_inception()
inception_model.summary()

In [ ]:
inception_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_inception = train_and_evaluate(
    inception_model, 'custom_inception',
    train_dataset, val_dataset, test_dataset
)
results.append(result_inception)

**Observations (Custom Inception):**

_TODO: Comment on the multi-scale feature extraction. How does this compare with the ResNet and improved CNN? Discuss parameter efficiency._

---
## 9. Results Comparison

Summary of all custom CNN models trained.

In [ ]:
import pandas as pd

# Build comparison table
comparison_df = pd.DataFrame([
    {
        'Model': r['model_name'],
        'Test Accuracy': f"{r['test_accuracy']:.4f}",
        'Test Loss': f"{r['test_loss']:.4f}"
    }
    for r in results
])

print("\n" + "="*60)
print("RESULTS SUMMARY - Custom CNNs")
print("="*60)
display(comparison_df)

In [ ]:
# Bar chart comparison
model_names = [r['model_name'] for r in results]
test_accs = [r['test_accuracy'] for r in results]

plt.figure(figsize=(10, 5))
bars = plt.bar(model_names, test_accs, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.ylabel('Test Accuracy')
plt.title('Custom CNN Models - Test Accuracy Comparison')
plt.ylim(0, 1)
for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Conclusions

_TODO: Write a comprehensive comparison of all custom CNN models:_

- _Which model achieved the best accuracy and why?_
- _How did regularization techniques (data augmentation, dropout, batch normalization) affect the results?_
- _What are the advantages and disadvantages of each architecture (simple CNN vs ResNet vs Inception)?_
- _What are the main challenges when working with a small dataset like STL-10?_
- _What improvements could be made in future iterations?_